# Archetype clustering of Bolivian energy regions

Groups Bolivia's 21 energy regions (8 connected to the National Interconnected System and 13 isolated microgrids) into representative clusters for EnergyScope Multicell Pathway. This notebook reproduces Sections 2.1 and 4.1 of the paper and Section B of the Supplementary Material.

**Workflow**
1. Standardize and weight the 13 regional attributes and build the hybrid (feature + geographic) distance matrix.
2. Select the number of clusters $K^*$ as the rounded mean of the Elbow, Silhouette, and Gap statistic optima (Table 6).
3. Run K-medoids with $K^*$.
4. Plot the cluster map (Figure 5).
5. Summarize the cluster characteristics (Table 7).

**Folder layout**
```
Archetype_clustering/
├── clustering_regions.ipynb
├── requirements.txt
├── inputs/
│   ├── regions_database.csv
│   └── division_SA.geojson
└── outputs/            (created on the first run)
```
Run the notebook from the `Archetype_clustering` folder. In Google Colab, the setup cells clone the repository to get the input files.

## 0. Setup

In [ ]:
# Check dependencies. If they are missing (e.g., in Colab), install them and restart the kernel.
import subprocess
import sys


def dependencies_ok():
    # scikit-learn-extra is built against numpy 1.x (numpy must be < 2.0)
    # and needs setuptools to provide distutils on Python >= 3.12
    import numpy as np
    if int(np.__version__.split('.')[0]) >= 2:
        return False
    try:
        from sklearn_extra.cluster import KMedoids  # noqa: F401
    except Exception:
        return False
    return True


if dependencies_ok():
    print('Dependencies OK.')
else:
    print('Installing scikit-learn-extra, numpy<2.0, and setuptools...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'scikit-learn-extra', 'numpy<2.0', 'setuptools'], check=True)
    print('Restarting the kernel. Run the notebook again after the restart.')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)  # True = restart

In [ ]:
import os
import subprocess
import sys
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from geopy.distance import geodesic
from matplotlib.lines import Line2D
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.preprocessing import StandardScaler
from sklearn_extra.cluster import KMedoids

warnings.filterwarnings('ignore')

# Paths, relative to the Archetype_clustering folder
INPUT_DIR = 'inputs'
OUTPUT_DIR = 'outputs'
REPO_URL = 'https://github.com/CIE-UMSS/EnergyScope_Multicell_Pathway_BO.git'
REPO_DIR = 'EnergyScope_Multicell_Pathway_BO'

# In Colab, clone the repository to get the input files
if 'google.colab' in sys.modules and not os.path.isdir(INPUT_DIR):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)
    os.chdir(os.path.join(REPO_DIR, 'Archetype_clustering'))

os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')

## 1. Data preprocessing and hybrid distance

Continuous variables are Z-score standardized and binary variables keep their 0/1 values (Eq. B.1). Each variable is multiplied by its weight (Table 5, Eq. B.2). The hybrid distance combines the normalized weighted Euclidean distance and the normalized geodesic distance between region centroids (Eqs. B.3–B.5).

In [ ]:
RANDOM_STATE = 42
ALPHA = 0.10  # Weight of the geographic distance (Eq. B.5)

# Variable weights by dimension (Table 5); they sum to 1 - ALPHA = 0.90.
# Keep this order: it sets the column order of the Gap statistic reference data.
WEIGHTS_BY_DIMENSION = {
    'Energy infrastructure': {
        'EL_network_access': 0.20,          # Power grid access (binary)
        'Gas_turbine': 0.20,                # Gas turbine presence (binary)
        'Diesel_genset': 0.12,              # Diesel genset presence (binary)
        'FG_network_access': 0.01,          # Gas network access (binary)
        'Num_vehicles_scaled': 0.02,        # Vehicles per capita
    },
    'Geographic & meteorological': {
        'Temp_scaled': 0.12,                # Annual average ambient temperature
        'Altitude_scaled': 0.10,            # Average altitude
        'Relative_humidity_scaled': 0.03,   # Annual average relative humidity
    },
    'Energy resources': {
        'Cp_PV_scaled': 0.035,              # Annual PV capacity factor
        'Cp_Wind_scaled': 0.035,            # Annual wind capacity factor
    },
    'Socioeconomic': {
        'GDP_scaled': 0.01,                 # GDP per capita
        'Population_density_scaled': 0.01,  # Population density
    },
    'Energy demand': {
        'EL_demand_scaled': 0.01,           # Annual electricity demand per capita
    },
}
WEIGHTS = {var: w for group in WEIGHTS_BY_DIMENSION.values() for var, w in group.items()}

CONTINUOUS_VARS = ['Population_density', 'Cp_PV', 'Cp_Wind', 'Temp', 'Relative_humidity',
                   'Altitude', 'EL_demand', 'GDP', 'Num_vehicles']
BINARY_VARS = ['FG_network_access', 'EL_network_access', 'Gas_turbine', 'Diesel_genset']

# Weight check (Table 5)
print('Weights by dimension:')
for dimension, group in WEIGHTS_BY_DIMENSION.items():
    print(f'  {dimension:32s}{len(group):3d} variables   {sum(group.values()):.2f}')
print(f"  {'Spatial (geographic distance)':32s}{'':15s}{ALPHA:.2f}")
total_weight = sum(WEIGHTS.values()) + ALPHA
print(f"  {'Total':32s}{'':15s}{total_weight:.2f}")
if abs(total_weight - 1.0) > 1e-3:
    raise ValueError(f'Weights must sum to 1.0 (current sum: {total_weight:.4f}).')

# Load data
data = pd.read_csv(os.path.join(INPUT_DIR, 'regions_database.csv'), sep=';')
required = ['Region', 'Latitude', 'Longitude', 'Number_municipalities'] + CONTINUOUS_VARS + BINARY_VARS
missing = [col for col in required if col not in data.columns]
if missing:
    raise KeyError(f'Missing columns in regions_database.csv: {missing}')
print(f'\nLoaded {len(data)} regions.')

# Standardize continuous variables (Eq. B.1); binary variables are not scaled
for var in CONTINUOUS_VARS:
    data[f'{var}_scaled'] = StandardScaler().fit_transform(data[[var]])

# Weighted feature matrix (Eq. B.2)
X = data[list(WEIGHTS)].to_numpy() * np.array(list(WEIGHTS.values()))


def geodesic_distance_matrix(df):
    # Pairwise geodesic distances [km] on the WGS-84 ellipsoid
    coords = list(zip(df['Latitude'], df['Longitude']))
    n = len(coords)
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            dist[i, j] = dist[j, i] = geodesic(coords[i], coords[j]).km
    return dist


geo_dist = geodesic_distance_matrix(data)
euc_dist = euclidean_distances(X)

# Normalize both matrices to [0, 1] (Eq. B.4) and combine them (Eq. B.5)
geo_norm = geo_dist / (geo_dist.max() + 1e-10)
euc_norm = euc_dist / (euc_dist.max() + 1e-10)
D = ALPHA * geo_norm + (1 - ALPHA) * euc_norm

print(f'Geodesic distances: {geo_dist[geo_dist > 0].min():.1f} to {geo_dist.max():.1f} km')
print(f'Hybrid distance matrix: {D.shape[0]} x {D.shape[1]} (alpha = {ALPHA:.2f})')

## 2. Optimal number of clusters

$K$ is evaluated from 2 to 20. Inertia and Silhouette are averaged over 100 K-medoids initializations per $K$.

- **Elbow** (Eq. B.6): point of the normalized inertia curve farthest from the line joining its endpoints.
- **Silhouette** (Eqs. B.7–B.8): $K$ with the highest mean Silhouette score.
- **Gap statistic** (Eqs. B.10–B.11): smallest $K$ with $Gap_K \geq Gap_{K+1} - s_{K+1}$, using $B = 20$ reference datasets.
- **Consensus** (Eq. B.12): $K^* = \mathrm{round}\left((K_{elbow} + K_{silhouette} + K_{gap})/3\right)$.

In [ ]:
K_MIN, K_MAX = 2, 20
N_INIT = 100  # K-medoids initializations per K
N_REFS = 20   # Reference datasets for the Gap statistic (B)

k_values = list(range(K_MIN, min(K_MAX, len(data) - 1) + 1))


def kmedoids(k, seed):
    return KMedoids(n_clusters=k, metric='precomputed', init='k-medoids++',
                    max_iter=300, random_state=seed)


def elbow_k(k_values, inertias):
    # Normalize the curve to [0, 1] and pick the point with the largest
    # perpendicular distance to the line joining the first and last points
    x = np.array(k_values, dtype=float)
    y = np.array(inertias, dtype=float)
    x_n = (x - x.min()) / (x.max() - x.min() + 1e-10)
    y_n = (y - y.min()) / (y.max() - y.min() + 1e-10)
    p1, p2 = (x_n[0], y_n[0]), (x_n[-1], y_n[-1])
    cross = np.abs((p2[0] - p1[0]) * (p1[1] - y_n) - (p1[0] - x_n) * (p2[1] - p1[1]))
    length = np.sqrt((p2[0] - p1[0]) ** 2 + (p2[1] - p1[1]) ** 2)
    return int(x[np.argmax(cross / (length + 1e-10))])


def silhouette_k(k_values, scores):
    return k_values[int(np.argmax(scores))]


def gap_k(k_values, gaps, s_k):
    # Tibshirani criterion: smallest K with Gap(K) >= Gap(K+1) - s(K+1)
    for i in range(len(gaps) - 1):
        if gaps[i] >= gaps[i + 1] - s_k[i + 1]:
            return k_values[i]
    return k_values[int(np.argmax(gaps))]


def gap_statistic(X, D, k_values, n_refs, seed):
    # Gap(K) = mean_b[log(W*_Kb)] - log(W_K),  s_K = sd_K * sqrt(1 + 1/B)
    # W_K:   K-medoids inertia on the hybrid distance matrix D.
    # W*_Kb: K-medoids inertia on reference data sampled uniformly within the
    #        observed range of each weighted variable (normalized Euclidean distance).
    np.random.seed(seed)
    gaps, s_k = [], []
    for k in k_values:
        try:
            log_w = np.log(kmedoids(k, seed).fit(D).inertia_ + 1e-10)
        except Exception:
            gaps.append(0)
            s_k.append(0)
            continue

        ref_log_w = []
        for b in range(n_refs):
            X_ref = np.random.uniform(low=X.min(axis=0), high=X.max(axis=0), size=X.shape)
            try:
                d_ref = euclidean_distances(X_ref)
                d_ref = d_ref / (d_ref.max() + 1e-10)
                ref_log_w.append(np.log(kmedoids(k, seed + b).fit(d_ref).inertia_ + 1e-10))
            except Exception:
                continue

        if not ref_log_w:
            gaps.append(0)
            s_k.append(0)
            continue

        ref_log_w = np.array(ref_log_w)
        gaps.append(ref_log_w.mean() - log_w)
        s_k.append(np.sqrt(np.mean((ref_log_w - ref_log_w.mean()) ** 2)) * np.sqrt(1 + 1 / n_refs))
    return gaps, s_k


# Inertia and Silhouette, averaged over N_INIT initializations
print('Inertia and Silhouette, K =', end=' ')
inertia, silhouette = [], []
for k in k_values:
    print(k, end=' ', flush=True)
    inertia_k, silhouette_k_runs = [], []
    for i in range(N_INIT):
        try:
            model = kmedoids(k, RANDOM_STATE + i)
            labels = model.fit_predict(D)
            inertia_k.append(model.inertia_)
            silhouette_k_runs.append(silhouette_score(D, labels, metric='precomputed'))
        except Exception:
            continue
    inertia.append(np.mean(inertia_k) if inertia_k else np.nan)
    silhouette.append(np.mean(silhouette_k_runs) if silhouette_k_runs else np.nan)

print('\nGap statistic...')
gaps, s_k = gap_statistic(X, D, k_values, N_REFS, RANDOM_STATE)

# Optimal K by method and consensus (Eq. B.12)
k_elbow = elbow_k(k_values, inertia)
k_silhouette = silhouette_k(k_values, silhouette)
k_gap = gap_k(k_values, gaps, s_k)
k_mean = np.mean([k_elbow, k_silhouette, k_gap])
k_consensus = int(round(k_mean))  # The mean of three integers never ends in .5

print(f'\nOptimal K: Elbow = {k_elbow}, Silhouette = {k_silhouette}, Gap statistic = {k_gap}')
print(f'Consensus: K* = round(({k_elbow} + {k_silhouette} + {k_gap}) / 3) = '
      f'round({k_mean:.2f}) = {k_consensus}')

# Table 6
methods = [','.join(tag for tag, k_opt in (('E', k_elbow), ('S', k_silhouette), ('G', k_gap))
                    if k_opt == k) or '-' for k in k_values]
metrics = pd.DataFrame({'K': k_values, 'Inertia': inertia, 'Silhouette': silhouette,
                        'Gap': gaps, 's_K': s_k, 'Method': methods})
print('\nClustering metrics (E = Elbow, S = Silhouette, G = Gap statistic):')
print(metrics.to_string(index=False, formatters={'Inertia': '{:.2f}'.format,
                                                 'Silhouette': '{:.3f}'.format,
                                                 'Gap': '{:.3f}'.format,
                                                 's_K': '{:.3f}'.format}))
metrics.to_csv(os.path.join(OUTPUT_DIR, 'optimal_clusters_metrics.csv'), index=False)

# Figure
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))


def mark_optimum(ax, values, k_opt):
    ax.axvline(k_opt, color='#E74C3C', linestyle='--', linewidth=2, label=f'Optimum: K = {k_opt}')
    ax.axvline(k_consensus, color='#2ECC71', linewidth=3, alpha=0.5, label=f'Consensus: K* = {k_consensus}')
    ax.scatter([k_opt], [values[k_values.index(k_opt)]], color='#E74C3C', s=200, zorder=5,
               edgecolor='black', linewidth=2)


axes[0].plot(k_values, inertia, 'o-', linewidth=2, markersize=8, color='#3498DB')
mark_optimum(axes[0], inertia, k_elbow)
axes[0].set_ylabel('Inertia', fontsize=11, weight='bold')
axes[0].set_title('Elbow method\n(maximum perpendicular distance)', fontsize=12, weight='bold')

axes[1].plot(k_values, silhouette, 'o-', linewidth=2, markersize=8, color='#E67E22')
mark_optimum(axes[1], silhouette, k_silhouette)
axes[1].set_ylabel('Silhouette score', fontsize=11, weight='bold')
axes[1].set_title('Silhouette score\n(maximum value)', fontsize=12, weight='bold')

axes[2].errorbar(k_values, gaps, yerr=s_k, marker='o', linewidth=2, markersize=8,
                 capsize=5, capthick=2, color='#9B59B6', label=r'Gap $\pm$ $s_K$')
mark_optimum(axes[2], gaps, k_gap)
axes[2].set_ylabel('Gap statistic', fontsize=11, weight='bold')
axes[2].set_title('Gap statistic\n(Tibshirani criterion)', fontsize=12, weight='bold')

for ax in axes:
    ax.set_xticks(range(k_values[0], k_values[-1] + 1, 2))
    ax.set_xlabel('Number of clusters (K)', fontsize=11, weight='bold')
    ax.legend(fontsize=9)

fig.suptitle(f'Optimal number of clusters (hybrid distance, alpha = {ALPHA:.2f}): '
             f'K* = {k_consensus}', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'optimal_clusters_evaluation.png'), dpi=300, bbox_inches='tight')
plt.show()

## 3. K-medoids clustering

K-medoids with $K^*$ clusters on the hybrid distance matrix (Eq. B.13), initialized with k-medoids++. The medoid of each cluster is the region used as its modeling archetype. Clusters are labeled C1 to C$K^*$ as in the paper.

In [ ]:
K_CLUSTERS = k_consensus  # K* = 6 in the paper

model = kmedoids(K_CLUSTERS, RANDOM_STATE)
data['Cluster'] = model.fit_predict(D) + 1
data['Is_medoid'] = np.isin(np.arange(len(data)), model.medoid_indices_)

sil = silhouette_score(D, data['Cluster'], metric='precomputed')
print(f'K = {K_CLUSTERS} | inertia = {model.inertia_:.4f} | Silhouette = {sil:.4f}\n')

for c in sorted(data['Cluster'].unique()):
    members = data[data['Cluster'] == c]
    medoid = members.loc[members['Is_medoid'], 'Region'].iloc[0]
    print(f"C{c} | medoid: {medoid:<11s}| {len(members)} regions "
          f"({members['Number_municipalities'].sum()} municipalities): {', '.join(members['Region'])}")

results = data[['Region', 'Cluster', 'Is_medoid', 'Number_municipalities',
                'EL_network_access', 'FG_network_access', 'Gas_turbine', 'Diesel_genset',
                'Population_density', 'Cp_PV', 'Cp_Wind', 'Temp', 'Altitude',
                'EL_demand', 'GDP']].sort_values(['Cluster', 'Region'])
results.to_csv(os.path.join(OUTPUT_DIR, 'clustering_results.csv'), index=False)

## 4. Cluster map (Figure 5)

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

REGION_COLUMN = 'DEPARTAMEN'  # Region name field in the GeoJSON; must match 'Region' in the CSV
CLUSTER_COLORS = {1: '#66C2A5', 2: '#8DA0CB', 3: '#E78AC3', 4: '#A6D854', 5: '#FFD92F', 6: '#FC8D62'}
UNASSIGNED_COLOR = '#D9D9D9'
DISPLAY_NAMES = {'Potosi': 'Potosí'}
LABEL_OFFSETS = {'Beni': (0.8, -0.3), 'Tarija': (0.3, 0.05), 'SA-BE': (-0.1, 0.15)}  # (lon, lat) in degrees

regions_map = gpd.read_file(os.path.join(INPUT_DIR, 'division_SA.geojson'))
gmap = regions_map.merge(data[['Region', 'Cluster']], left_on=REGION_COLUMN, right_on='Region', how='left')

unmatched = gmap.loc[gmap['Cluster'].isna(), REGION_COLUMN].tolist()
if unmatched:
    print(f'Warning: map regions without a cluster (check names): {unmatched}')
gmap['Cluster'] = gmap['Cluster'].fillna(0).astype(int)  # 0 = unassigned
gmap['Color'] = gmap['Cluster'].map(lambda c: CLUSTER_COLORS.get(c, UNASSIGNED_COLOR))

fig, ax = plt.subplots(figsize=(10, 10))
gmap.plot(ax=ax, color=gmap['Color'], alpha=0.85, edgecolor='black', linewidth=1.2)

minx, miny, maxx, maxy = gmap.total_bounds
ax.set_xlim(minx - 0.1, maxx + 0.1)
ax.set_ylim(miny - 0.1, maxy + 0.1)

# Region name and cluster label at each centroid
for _, row in gmap.iterrows():
    if row.geometry is None:
        continue
    name = row[REGION_COLUMN]
    dx, dy = LABEL_OFFSETS.get(name, (0, 0))
    cluster_label = f"C{row['Cluster']}" if row['Cluster'] > 0 else 'N/A'
    ax.text(row.geometry.centroid.x + dx, row.geometry.centroid.y + dy,
            f'{DISPLAY_NAMES.get(name, name)}\n{cluster_label}',
            fontsize=10, ha='center', va='center', weight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='black', linewidth=1.0, alpha=0.35))

handles = [Line2D([0], [0], marker='s', color='w', label=f'Cluster {c} (C{c})',
                  markerfacecolor=CLUSTER_COLORS.get(c, UNASSIGNED_COLOR), markersize=12,
                  alpha=0.85, markeredgecolor='black', markeredgewidth=1)
           for c in sorted(data['Cluster'].unique())]
ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.0, 1.0), frameon=True,
          edgecolor='gray', facecolor='white', fontsize=12, framealpha=0.95,
          borderpad=0.5, handletextpad=0.5)
ax.axis('off')

plt.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.02)
plt.savefig(os.path.join(OUTPUT_DIR, 'clustering_map.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'clustering_map.pdf'), format='pdf', bbox_inches='tight')
plt.show()

## 5. Cluster characteristics (Table 7)

Unweighted means over the regions of each cluster. Infrastructure columns give the share of regions with access or presence.

In [ ]:
grouped = data.groupby('Cluster')
table7 = pd.DataFrame({
    'Medoid': data[data['Is_medoid']].set_index('Cluster')['Region'],
    'N_reg': grouped.size(),
    'N_mun': grouped['Number_municipalities'].sum(),
    'EL (%)': grouped['EL_network_access'].mean() * 100,
    'FG (%)': grouped['FG_network_access'].mean() * 100,
    'GT (%)': grouped['Gas_turbine'].mean() * 100,
    'DG (%)': grouped['Diesel_genset'].mean() * 100,
    'D_el (MWh/person/y)': grouped['EL_demand'].mean(),
    'GDP (USD/person)': grouped['GDP'].mean(),
    'T_avg (C)': grouped['Temp'].mean(),
    'h_avg (m.a.s.l.)': grouped['Altitude'].mean(),
    'Cp_PV (%)': grouped['Cp_PV'].mean(),
    'Cp_Wind (%)': grouped['Cp_Wind'].mean(),
})
table7.index = [f'C{c}' for c in table7.index]
table7.index.name = 'Cluster'
table7 = table7.round({'EL (%)': 0, 'FG (%)': 0, 'GT (%)': 0, 'DG (%)': 0,
                       'D_el (MWh/person/y)': 3, 'GDP (USD/person)': 0, 'T_avg (C)': 1,
                       'h_avg (m.a.s.l.)': 0, 'Cp_PV (%)': 1, 'Cp_Wind (%)': 1})

print(table7.to_string())
table7.to_csv(os.path.join(OUTPUT_DIR, 'cluster_summary.csv'))